
# Benchmark GCN on LS


### Introduction

This notebook walks through reproducing the key results from the paper.
**Before you begin**, please download the processed datasets from Zenodo and place them under `data/processed_data/gcn_processed_data/`:

> https://zenodo.org/records/15230565

Once the data files are in place, you can run each cell below to regenerate the benchmark metrics and plots exactly as reported.


## 1. Import Dependencies

First, we import the necessary dependencies for our evaluation.

In [1]:
# Suppress PyTorch Warnings and ensure proper dependency loading
import warnings
import os
import sys

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) # Go two levels up from the notebook location to reach project root

if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root set to:", project_root)

Project root set to: /Users/SQuASH


In [2]:
import os
import torch
import json
from torch_geometric.data import DataLoader
import numpy as np

from config import DeviceConfig, get_default_model_config_by_search_space, PathConfig, QCConfig, \
    get_model_config_from_path
from evaluate.evaluate_model import run_gcn_evaluation
from util.config_utils import get_gate_set_and_features_by_name
from util.data_loader import load_data
from surrogate_models.architectures.gnn.gcn_runner import prepare_paths_and_config, set_seed
from surrogate_models.architectures.gnn.gnn_model import RegGNN
from evaluate.evaluate_utils import compute_metrics
from evaluate.evaluate_utils import load_gnn_model, evaluate_gnn


## 2. Setup Configuration


We begin by choosing the specific search space in which we’ll reproduce the benchmark results.

In [3]:
# Define the search space
search_space = 'ls_a'

Next, we define paths and other configurations necessary for loading the model and dataset.


In [4]:
# Initialize device configuration and path configuration
device_config = DeviceConfig()
device        = device_config.device
path_config   = PathConfig()

# also load the full JSON + timestamp if you like
config, gate_set_name, timestamp = prepare_paths_and_config(search_space, device)
print(f"\n=== Search‐space: {search_space} ({timestamp}) ===")
print("Config:")
print(json.dumps(config, indent=2, default=str))


=== Search‐space: ls_a (2025-05-30_12-49-16) ===
Config:
{
  "device": "cpu",
  "seed": 42,
  "runseed": 42,
  "batch_size": 32,
  "num_workers": 0,
  "epochs": 100,
  "emb_dim": 1050,
  "layer_num": 8,
  "qubit_num": 4,
  "num_node_features": 8,
  "drop_ratio": 0.0644893118913786,
  "lr": 4.540520885756229e-05,
  "decay": 1.917208797826118e-06,
  "JK": "mean",
  "patience": 7,
  "metric": "spearman",
  "graph_pooling": "attention",
  "n_estimators": null,
  "max_depth": null,
  "random_state": null,
  "optuna_trials": null,
  "min_samples_split": null,
  "min_samples_leaf": null,
  "max_features": null,
  "n_jobs": null,
  "PATHS": {
    "optuna_studies": "/Users/SQuASH/surrogate_models/tuning/studies",
    "raw_data": "/Users/SQuASH/data/raw_data/",
    "gcn_data": "/Users/SQuASH/data/processed_data/gcn_processed_data",
    "rf_data": "/Users/SQuASH/data/processed_data/rf_processed_data",
    "trained_models": "/Users/SQuASH/surrogate_models/trained_models",
    "benchmark_search_sp

### 🔧 Step 3: Set Random Seed


In [5]:
set_seed(config["runseed"])

## 4. Load Datasets and Benchmark GCN

In [6]:
dataset_names = [
    f"{search_space}_squash",
]
model_names = [
    f"gcn_{search_space}",
]

# Loop over each dataset/model pair
for data_set, model_name in zip(dataset_names, model_names):
    print(f"\n=== Dataset: {data_set} with Model: {model_name} ===")

    # load the data
    data_path = os.path.join(
        path_config.paths['gcn_data'],
        f"graph_data_{search_space}",
        f"{data_set}.pt"
    )
    circuits   = torch.load(data_path, weights_only=False)          # list of Data objects

    loader     = DataLoader(circuits, batch_size=32, shuffle=False)

    # load model config
    cfg_path   = os.path.join(
        os.path.join(path_config.paths['benchmark_search_spaces'], f'{search_space}/surrogate_models/configs', f'{model_name}_config.json')
    )
    model_path = os.path.join(path_config.paths['benchmark_search_spaces'], f'{search_space}/surrogate_models', f"{model_name}.pth")
    model_cfg  = get_model_config_from_path(cfg_path, device)

    # instantiate RegGNN
    model = load_gnn_model(model_path, model_cfg)
    model.to(device).eval()

    # inference
    preds, labels  = evaluate_gnn(circuits, model, model_cfg)

    # compute and print metrics
    compute_metrics(preds, labels, label=model_name, tolerance=0.1)


=== Dataset: ls_a_squash with Model: gcn_ls_a ===
--- Metrics for gcn_ls_a ---
Samples: 17285
MSE:     0.0026
MAE:     0.0318
RMSE:    0.0510
R^2:     0.8927
Corr:    0.9470
Spearman: 0.9284
Accuracy (|err| <= 0.1): 94.10%

